In [2]:
# notebook: 01_cmvts_extension_predictor_outcome_validation.ipynb
# ============================================================================
# CMVTS Extension Study — Predictor–Outcome Validation (multi-country)
# ----------------------------------------------------------------------------
# Purpose:
#   Break the Appendix-B circularity of the original paper by validating that a
#   PREDICTOR (macro-structural similarity, Korea vs target) explains an
#   independently-measured OUTCOME (realized behavioral divergence from Findex).
#
# Design rules (do NOT violate — this is what defeats circularity):
#   1. SOURCE behavioral distribution  -> ONLY from Korean CB data (2022-12).
#   2. OUTCOME (realized JSD)           -> ONLY from target-country Findex 2024
#                                          behavior-STRENGTH vars (card/digital use).
#   3. PREDICTOR (macro CosSim/Spearman)-> from a DISJOINT variable group
#                                          (demographics/account-structure/saving/
#                                          borrowing) — never the outcome vars.
#   The predictor and outcome must never share input variables.
#
# Time-alignment (design choice C):
#   Compute the macro predictor at BOTH 2022 and 2024 vintages and report the
#   sensitivity. This turns the source(2022)/Findex(2024) time gap into the
#   dynamic-indicator sensitivity analysis requested by the reviewer.
# ============================================================================

import os
import numpy as np
import pandas as pd
from scipy import stats

# ----------------------------------------------------------------------------
# 0. CONFIG — adjust paths to your local machine
# ----------------------------------------------------------------------------
# Korean CB synthetic files (quarter-end snapshots). Newest = 2022-12.
CB_DIR   = "."                              # directory holding the *_개인CB.csv files
CB_FILES = {
    "2019-12": "201912_개인CB.csv",
    "2020-12": "202012_개인CB.csv",
    "2021-12": "202112_개인CB.csv",
    "2022-12": "202212_개인CB.csv",          # <- source snapshot used for the scorecard
}
CB_SOURCE_KEY = "2022-12"                    # source vintage for WoE / scorecard

# Findex 2025 microdata (2024 collection), labelled CSV.
FINDEX_CSV = "findex_microdata_2025_labelled_update112425.csv"

# Target markets (Korea = source). Names must match Findex 'economy' column.
SOURCE_ECON = "Korea, Rep."
TARGET_ECONS = ["Indonesia", "Thailand", "Viet Nam", "Philippines",
                "Bangladesh", "Cambodia", "Nepal", "Pakistan", "Lao PDR"]

# Findex value convention: 1 = "yes"; 2 = "no"; 3/4 = don't know/refused (-> treat as no).
YES = 1

# CB sentinel/missing codes to scrub (from the original paper's data-quality step).
CB_SENTINELS = [8888888.8, -9, -99999999]

# CB alternative / card variables actually used in the original 9-variable transfer
# set. Adjust these names to the exact column headers in your CB CSV.
CB_CARD_VARS = ["C1M2B4W03", "C1M2B5W03", "C1Z001373"]        # 3-month card spend vars
CB_ALT_VARS  = ["AL012G019", "AL012G005", "AS120G001", "AL012G011"]

# ----------------------------------------------------------------------------
# 1. HELPERS
# ----------------------------------------------------------------------------
def jsd(p, q, eps=1e-12):
    """Jensen–Shannon divergence in bits (base-2). Bounded in [0, 1]."""
    p = np.asarray(p, float) + eps
    q = np.asarray(q, float) + eps
    p /= p.sum(); q /= q.sum()
    m = 0.5 * (p + q)
    kl = lambda a, b: np.sum(a * np.log2(a / b))
    return 0.5 * kl(p, m) + 0.5 * kl(q, m)

def wshare(g, var, yes=YES):
    """Weighted share of respondents with var == yes (unconditional over the sample)."""
    w = g["wgt"]
    return w[g[var] == yes].sum() / w.sum()

def safe_cols(df, cols):
    """Return only the columns that actually exist (guards against header drift)."""
    present = [c for c in cols if c in df.columns]
    missing = [c for c in cols if c not in df.columns]
    if missing:
        print(f"  [warn] missing columns skipped: {missing}")
    return present

# ============================================================================
# PART A — SOURCE behavioral distribution from Korean CB (2022-12)
# ============================================================================
# We build the source behavioral-intensity distribution from the CB CARD-SPEND
# variables. This is the fix for the trap found in the pilot: Korea's card vars
# are MISSING in Findex, so the source distribution must come from CB, not Findex.
# ----------------------------------------------------------------------------
def load_cb(path):
    """Load a CB snapshot, scrub sentinels to NaN."""
    df = pd.read_csv(os.path.join(CB_DIR, path), low_memory=False)
    df = df.replace(CB_SENTINELS, np.nan)
    return df

def cb_engagement_distribution(df_cb, n_bins=4):
    """
    Build a source behavioral-intensity distribution (n_bins) from CB card-spend.
    Method: sum available card-spend variables into a single intensity score,
    rank into quantile bins, return the (unweighted) bin-mass vector.
    Rationale: mirrors the WoE 'behavioral engagement' the scorecard rewards,
    while staying comparable to the Findex-derived target engagement bins.
    """
    cols = safe_cols(df_cb, CB_CARD_VARS)
    if not cols:
        raise ValueError("No CB card variables found — check CB_CARD_VARS names.")
    intensity = df_cb[cols].fillna(0).sum(axis=1)
    # quantile binning; if too few distinct values, fall back to rank binning
    try:
        bins = pd.qcut(intensity.rank(method="first"), q=n_bins, labels=False)
    except ValueError:
        bins = pd.cut(intensity, bins=n_bins, labels=False)
    dist = np.array([(bins == k).sum() for k in range(n_bins)], float)
    return dist / dist.sum()

# ============================================================================
# PART B — OUTCOME: realized behavioral divergence from Findex (target, 2024)
# ============================================================================
# OUTCOME uses ONLY behavior-strength vars: fin8 (card use), merchantpay_dig,
# anydigpayment. Same n_bins as the CB source distribution so JSD is defined.
# ----------------------------------------------------------------------------
OUTCOME_VARS = ["fin8", "merchantpay_dig", "anydigpayment"]

def findex_engagement_distribution(g, n_bins=4):
    """Weighted engagement-count distribution (0..n_bins-1) from OUTCOME_VARS."""
    cols = safe_cols(g, OUTCOME_VARS)
    sig = pd.DataFrame({c: (g[c] == YES).astype(int) for c in cols})
    score = sig.sum(axis=1).clip(upper=n_bins - 1)
    w = g["wgt"]
    d = np.array([w[score == k].sum() for k in range(n_bins)], float)
    return d / d.sum()

# ============================================================================
# PART C — PREDICTOR: macro-structural similarity (disjoint variable group)
# ============================================================================
# PREDICTOR must NOT use any OUTCOME_VARS. Uses account structure / saving /
# borrowing / wages / workforce. Cosine similarity of the Korea vs target vector.
# The same predictor is computable at different macro vintages for sensitivity.
# ----------------------------------------------------------------------------
PREDICTOR_VARS = ["account_fin", "account_mob", "saved", "borrowed",
                  "receive_wages", "emp_in"]

def macro_vector(g):
    cols = safe_cols(g, PREDICTOR_VARS)
    return np.array([wshare(g, v, YES) for v in cols], float)

def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

# ============================================================================
# MAIN
# ============================================================================
print("=" * 70)
print("STEP 1 — Source behavioral distribution from Korean CB", CB_SOURCE_KEY)
print("=" * 70)
cb_src = load_cb(CB_FILES[CB_SOURCE_KEY])
print("CB source shape:", cb_src.shape)
src_dist = cb_engagement_distribution(cb_src)
print("Source engagement distribution (bins):", np.round(src_dist, 4))

print("\n" + "=" * 70)
print("STEP 2 — Load Findex 2024 targets")
print("=" * 70)
fx = pd.read_csv(FINDEX_CSV, low_memory=False)
print("Findex shape:", fx.shape, "| economies:", fx["economy"].nunique())

# source macro vector (Korea) from Findex — used for the predictor only
kr_macro = macro_vector(fx[fx["economy"] == SOURCE_ECON])

print("\n" + "=" * 70)
print("STEP 3 — Predictor vs Outcome per target")
print("=" * 70)
rows = []
for e in TARGET_ECONS:
    g = fx[fx["economy"] == e]
    if len(g) == 0:
        print(f"  [warn] {e} not found in Findex — skipped")
        continue
    # OUTCOME: realized JSD between CB source dist and target Findex dist
    y_jsd = jsd(src_dist, findex_engagement_distribution(g))
    # PREDICTOR: macro cosine similarity (Korea vs target)
    x_cos = cosine(kr_macro, macro_vector(g))
    rows.append({"economy": e,
                 "X_macro_cos": round(x_cos, 4),
                 "Y_realized_JSD": round(y_jsd, 4),
                 "Y_1_minus_JSD": round(1 - y_jsd, 4)})

res = pd.DataFrame(rows).sort_values("Y_realized_JSD").reset_index(drop=True)
print(res.to_string(index=False))

print("\n" + "=" * 70)
print("STEP 4 — Predictor–Outcome correlation (circularity-broken)")
print("=" * 70)
# Expected sign: higher macro similarity -> LOWER realized divergence (negative r).
# NOTE from pilot: if the source distribution is contaminated (e.g. Findex card
# vars missing for Korea), the sign can flip. Using CB for the source (as here)
# is what restores the expected direction. Watch the sign as a diagnostic.
r_p, p_p = stats.pearsonr(res["X_macro_cos"], res["Y_realized_JSD"])
r_s, p_s = stats.spearmanr(res["X_macro_cos"], res["Y_realized_JSD"])
print(f"Pearson  r = {r_p:+.3f}  (p = {p_p:.4f})")
print(f"Spearman r = {r_s:+.3f}  (p = {p_s:.4f})")
print("Interpretation: negative & significant => macro structure predicts")
print("realized behavioral divergence, supporting the CMVTS core assumption.")

# ============================================================================
# STEP 5 — Design-choice C: macro-vintage sensitivity (2022 vs 2024)
# ============================================================================
# Placeholder for the time-alignment sensitivity. Recompute the PREDICTOR using
# macro indicators at the 2022 vintage (e.g., Findex 2021 country-level values
# or GSMA 2022) and compare correlation strength/sign against the 2024 vintage.
# Fill MACRO_2022 with a dict {economy: np.array([...])} matching PREDICTOR_VARS
# order once you have the 2022 macro values, then uncomment.
# ----------------------------------------------------------------------------
# MACRO_2022 = { ... }
# rows22 = []
# kr22 = MACRO_2022[SOURCE_ECON]
# for e in TARGET_ECONS:
#     if e not in MACRO_2022: continue
#     x = cosine(kr22, MACRO_2022[e])
#     y = res.loc[res.economy == e, "Y_realized_JSD"].values[0]
#     rows22.append({"economy": e, "X_macro_cos_2022": round(x, 4), "Y_realized_JSD": y})
# res22 = pd.DataFrame(rows22)
# r22, p22 = stats.pearsonr(res22["X_macro_cos_2022"], res22["Y_realized_JSD"])
# print(f"\n[2022 vintage] Pearson r = {r22:+.3f} (p = {p22:.4f})")
# print("Compare against the 2024 vintage above -> dynamic-indicator sensitivity.")

# ============================================================================
# STEP 6 — Signed-divergence diagnostic (new methodological contribution)
# ============================================================================
# Unsigned JSD ignores DIRECTION of divergence (above vs below the source in
# engagement). Report a signed variant to expose this, per the pilot finding.
# ----------------------------------------------------------------------------
def findex_engagement_mean(g):
    cols = safe_cols(g, OUTCOME_VARS)
    sig = sum((g[c] == YES).astype(int) for c in cols)
    w = g["wgt"]
    return (sig * w).sum() / w.sum()

# source engagement mean from CB (not Findex) to keep the source consistent
src_cols = safe_cols(cb_src, CB_CARD_VARS)
src_intensity = cb_src[src_cols].fillna(0).sum(axis=1)
src_eng_rankmean = src_intensity.rank(pct=True).mean()  # scale-free reference

srows = []
for e in TARGET_ECONS:
    g = fx[fx["economy"] == e]
    if len(g) == 0:
        continue
    y = jsd(src_dist, findex_engagement_distribution(g))
    # crude sign: is target digital engagement below the (rank-normalized) source?
    sign = np.sign(0.5 - findex_engagement_mean(g) / 3.0)  # 3 = max engagement count
    srows.append({"economy": e, "signed_JSD": round(sign * y, 4)})
sres = pd.DataFrame(srows)
merged = res.merge(sres, on="economy")
print("\n" + "=" * 70)
print("STEP 6 — Signed vs unsigned divergence")
print("=" * 70)
print(merged.to_string(index=False))
r_signed, p_signed = stats.pearsonr(merged["X_macro_cos"], merged["signed_JSD"])
print(f"\nSigned-JSD Pearson r = {r_signed:+.3f} (p = {p_signed:.4f})")
print("If unsigned and signed correlations differ in sign, that is the")
print("evidence that C1 needs a directional formulation (paper contribution).")

STEP 1 — Source behavioral distribution from Korean CB 2022-12
CB source shape: (3129036, 157)
Source engagement distribution (bins): [0.25 0.25 0.25 0.25]

STEP 2 — Load Findex 2024 targets
Findex shape: (144090, 199) | economies: 140

STEP 3 — Predictor vs Outcome per target
    economy  X_macro_cos  Y_realized_JSD  Y_1_minus_JSD
   Thailand       0.8178          0.0203         0.9797
  Indonesia       0.7150          0.0673         0.9327
   Viet Nam       0.7575          0.0695         0.9305
      Nepal       0.6526          0.0894         0.9106
Philippines       0.5778          0.1137         0.8863
   Cambodia       0.6265          0.1220         0.8780
    Lao PDR       0.6752          0.1261         0.8739
 Bangladesh       0.5717          0.2032         0.7968
   Pakistan       0.4891          0.2176         0.7824

STEP 4 — Predictor–Outcome correlation (circularity-broken)
Pearson  r = -0.912  (p = 0.0006)
Spearman r = -0.867  (p = 0.0025)
Interpretation: negative & signif